In [2]:
import os
import re
import warnings
import random
from collections import defaultdict
from typing import Dict, List, Tuple

import torch
import torch.nn.functional as F
import numpy as np
import numpy as np
import torch
from tqdm.notebook import tqdm
from transformers import GPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings("ignore")

C:\Users\Uliana\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

## Задание

1) Реализовать методы `greedy_sampling` и `generate` (1 балл)
2) Реализовать метод `random_sampling` и поддержать его в `generate` (1 балл)
3) Реализовать метод `_beam_search_generate` и поддержать его в `generate` (2 балла)
4) Реализовать методы `apply_top_p`, `apply_top_k`, `apply_temperature` и поддержать их в `generate` (1 балл)  
Все методы необходимо реализовать через векторные операции в torch/numpy везде где это возможно

In [26]:
class Model:
    def __init__(self, model_name: str = "gpt2"):
        self.model = GPT2LMHeadModel.from_pretrained(model_name)
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.vocab_size = self.tokenizer.vocab_size

    def greedy_sampling(self, logits: torch.Tensor) -> int:
        return torch.argmax(logits, dim=-1).item()

    def random_sampling(self, logits: torch.Tensor) -> int:
        probs = F.softmax(logits, dim=-1)
        token_id = torch.multinomial(probs, num_samples=1).item()
        return token_id

    def _beam_search_generate(
        self,
        prompt: str,
        max_length: int,
        num_beams: int
    ) -> str:
        device = next(self.model.parameters()).device
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(device)
    
        beams = [(input_ids, 0.0)]
    
        for _ in range(max_length):
            new_beams = []
    
            for seq, score in beams:
                outputs = self.model(seq)
                logits = outputs.logits[:, -1, :]
                log_probs = torch.log_softmax(logits, dim=-1)
    
                # top num_beams кандидатов
                top_log_probs, top_ids = torch.topk(log_probs, num_beams, dim=-1)
    
                for i in range(num_beams):
                    next_token_id = top_ids[0, i].unsqueeze(0).unsqueeze(0)
                    new_seq = torch.cat([seq, next_token_id], dim=-1)
                    new_score = score + top_log_probs[0, i].item()
                    new_beams.append((new_seq, new_score))
    
            # сортируем и оставляем num_beams лучших
            beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:num_beams]
    
            if all(self.tokenizer.eos_token_id in beam[0] for beam in beams):
                break
    
        # нужна максимальная суммарной вероятность
        best_seq = max(beams, key=lambda x: x[1])[0]
        return self.tokenizer.decode(best_seq[0], skip_special_tokens=True)


    def apply_temperature(self, logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        return logits / temperature

    def _apply_top_p(self, logits: torch.Tensor, top_p: float = 1.0) -> torch.Tensor:
        if top_p >= 1.0:
            return logits

        probs = torch.softmax(logits, dim=-1)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0
    
        indices_to_remove = torch.zeros_like(logits, dtype=torch.bool)
        batch_indices = torch.arange(logits.size(0)).unsqueeze(-1)
        indices_to_remove[batch_indices, sorted_indices[sorted_indices_to_remove]] = True

        logits = logits.masked_fill(indices_to_remove, -float("inf"))
        return logits
        
    # def _apply_top_k(self, logits: torch.Tensor, top_k: float = self.vocab_size) -> torch.Tensor:
    def _apply_top_k(self, logits: torch.Tensor, top_k: float = None) -> torch.Tensor:
        if top_k is None:
            top_k = self.vocab_size
        
        if top_k <= 0 or top_k >= logits.size(-1):
            return logits
    
        values, _ = torch.topk(logits, top_k)
        min_value = values[..., -1, None]
    
        logits = torch.where(logits < min_value, torch.full_like(logits, -float("inf")), logits)
        return logits

    def generate(
        self,
        prompt: str,
        max_length: int = 50,
        strategy: str = "greedy",
        temperature: float = 1.0,
        top_k: int = 0,
        top_p: float = 1.0,
        num_beams: int = 3
    ) -> str:
        if strategy == "beam":
            return self._beam_search_generate(prompt, max_length, num_beams)
    
        device = next(self.model.parameters()).device
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(device)
    
        for _ in range(max_length):
            outputs = self.model(input_ids)
            logits = outputs.logits[:, -1, :]
    
            # temperature, top-k и top-p
            logits = self.apply_temperature(logits, temperature)
            logits = self._apply_top_k(logits, top_k)
            logits = self._apply_top_p(logits, top_p)
    
            # следующий токен
            if strategy == "greedy":
                next_token_id = self.greedy_sampling(logits)
            elif strategy == "random":
                next_token_id = self.random_sampling(logits)
            else:
                raise ValueError(f"Неизвестная стратегия: {strategy}")
    
            # добавляем токен к последовательности
            next_token_tensor = torch.tensor([[next_token_id]], device=device)
            input_ids = torch.cat([input_ids, next_token_tensor], dim=-1)
    
            # встречен EOS
            if next_token_id == self.tokenizer.eos_token_id:
                break

        return self.tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [27]:
model = Model("gpt2")

In [28]:
print(model.generate("Your future depends on", strategy="greedy", max_length=30))

Your future depends on it.

The first step is to get your business to understand the value of your product.

The second step is to understand the value


In [29]:
print(model.generate("Your future depends on", strategy="random", temperature=1.2, max_length=30))

Your future depends on what makes the allies crazier: what synchronicity dictates the real fractions of obstructionism. But on this aspect, China's tricky maneuver


In [30]:
print(model.generate("Your future depends on", strategy="random", top_k=50, max_length=30))

Your future depends on your ability to manage, maintain, and manage your businesses effectively".

One common question being raised is whether one cannot take the most effective use of


In [31]:
print(model.generate("Your future depends on", strategy="random", top_p=0.9, max_length=30))

Your future depends on those 5 things:

1. If Trump and the Wyden that gave him such a multi-million dollar special conference is acting as defender or


In [32]:
print(model.generate("Your future depends on", strategy="beam", num_beams=5, max_length=30))

Your future depends on it."

"I don't know what you're talking about. I don't know what you're talking about. I don't know what
